In [66]:
from dataclasses import dataclass
import numpy as np

@dataclass
class Grid:
    rows: int
    cols: int
    step_reward: int
    terminals: dict
    walls: set
    noise: float = 0.0
    actions = dict(up=(-1, 0), right=(0, 1), down=(1, 0), left=(0, -1))
    arrows = {'up': "↑", 'right': "→", 'down': "↓", 'left': "←"}
    def cell_repr(self, r, c):
        cell = (r, c)
        if cell in self.terminals:
            return self.terminals[cell]
        elif cell in self.walls:
            return '#'
        else:
            return '·'
    def render(self):
        for r in range(self.rows):
            for c in range(self.cols):
                if r == 0 and c == 0:
                    print(" r/c", end="")
                    print(''.join([f'{v:>3} ' for v in range(self.cols)]))
                if c == 0:
                    print(f'{r:>3} ', end="")
                print(f'{self.cell_repr(r, c):>3} ', end="")
            print()

    def step(self, cell, action):
        result = []
        action_keys = list(self.actions.keys())
        aind = list(self.actions.keys()).index(action)
        n_actions = len(action_keys)
        for a, prob in zip(
            [aind, (aind + 1) % n_actions, (aind - 1) % n_actions],
            [1 - self.noise, self.noise / 2, self.noise / 2],
        ):
            movement = self.actions[action_keys[a]]
            next_cell = (cell[0] + movement[0], cell[1] + movement[1])
            outside = not (0 <= next_cell[0] < self.rows and 0 <= next_cell[1] < self.cols)
            on_wall = next_cell in self.walls
            if outside or on_wall:
                next_cell = cell

            if next_cell in self.terminals:
                reward = self.terminals[next_cell]
            else:
                reward = self.step_reward
            result.append((prob, next_cell, reward))
        return result
    def properties(self):
        return f'''
Grid(
    rows={self.rows},
    cols={self.cols},
    step_reward={self.step_reward},
    terminals={self.terminals},
    walls={self.walls},
    noise={self.noise},
)
'''


default_grid = Grid(
    rows=3,
    cols=4,
    step_reward=-0.04,
    terminals={(0, 3): 1},
    walls={(1, 1)},
    noise=0.2,
)
default_grid

Grid(rows=3, cols=4, step_reward=-0.04, terminals={(0, 3): 1}, walls={(1, 1)}, noise=0.2)

In [67]:
default_grid.render()

 r/c  0   1   2   3 
  0   ·   ·   ·   1 
  1   ·   #   ·   · 
  2   ·   ·   ·   · 


In [68]:
class Sampler:
    def __init__(self, env: Grid, rng: np.random.Generator):
        self.env = env
        self.rng = rng
        self.empty_cells = [
            (r, c)
            for r in range(env.rows)
            for c in range(env.cols)
            if (r, c) not in env.terminals and (r, c) not in env.walls
        ]
        self.nS = len(self.empty_cells)

    def reset(self):
        i = self.rng.choice(len(self.empty_cells))
        return self.empty_cells[i]
    def step(self, cell, action):
        result = self.env.step(cell, action)
        i = self.rng.choice(len(result), p=[p for p, _, _ in result])
        _, next_cell, reward = result[i]
        done = True if next_cell in self.env.terminals else False
        return next_cell, reward, done

In [69]:
import io, contextlib, functools

def silent(fn):
    @functools.wraps(fn)
    def wrapper(*args, **kwargs):
        with contextlib.redirect_stdout(io.StringIO()):
            return fn(*args, **kwargs)
    return wrapper

In [70]:
def render_policy(grid: Grid, policy):
    for r in range(len(policy)):
        for c in range(len(policy[0])):
            if r == 0 and c == 0:
                print("r/c", end="")
                print(''.join([f'{v:>2} ' for v in range(grid.cols)]))
            if c == 0:
                print(f'{r:>2} ', end="")

            if policy[r][c] != None:
                value = grid.arrows[max(policy[r][c], key=policy[r][c].get)]
            else:
                value = grid.cell_repr(r, c)
            print(f' {value} ', end='')
        print()
    print('------------------')

def show_V(grid: Grid, V):
    for r in range(grid.rows):
        for c in range(grid.cols):
            if r == 0 and c == 0:
                print("  r/c", end="")
                print(''.join([f'{v:>4} ' for v in range(grid.cols)]))
            if c == 0:
                print(f'{r:>4} ', end="")
            print(f'{round(V[r][c], 2):>4} ', end="")
        print()
    print('------------------')


def value_iteration(grid: Grid, gamma=0.9, theta=1e-6, max_iters=1000):
    print('---------- value_iteration ------------')
    V = [[0 for _ in range(grid.cols)] for _ in range(grid.rows)]
    policy = [[None for _ in range(grid.cols)] for _ in range(grid.rows)]
    delta = float('inf')
    i = 0
    while delta > theta and i < max_iters:
        delta = 0.0
        V_old = [row[:] for row in V]
        for r in range(grid.rows):
            for c in range(grid.cols):
                cell = (r, c)
                if cell in grid.walls or cell in grid.terminals:
                    continue
                def q(action):
                    result = grid.step(cell, action)
                    q_value = 0
                    for prob, next_cell, reward in result:
                        q_value += prob * (
                            reward + gamma * V_old[next_cell[0]][next_cell[1]]
                        )
                    return q_value
                new_value = float('-inf')
                for action in grid.actions:
                    value = q(action)
                    if value > new_value:
                        new_value = value
                        policy[r][c] = {action: 1.0}
                V[r][c] = new_value
                delta = max(delta, abs(V[r][c] - V_old[r][c]))
        show_V(grid, V)
        i += 1
    render_policy(grid, policy)
    converged = delta <= theta
    if converged:
        print(f'value_iteration converged in {i} iterations')
    else:
        print(f'value_iteration did not converge in {max_iters} iterations')
    return policy, V, converged

In [71]:
policy, V, converged = value_iteration(default_grid);

---------- value_iteration ------------
  r/c   0    1    2    3 
   0 -0.04 -0.04 0.79    0 
   1 -0.04    0 -0.04 0.79 
   2 -0.04 -0.04 -0.04 -0.04 
------------------
  r/c   0    1    2    3 
   0 -0.08 0.52 0.86    0 
   1 -0.08    0  0.6 0.86 
   2 -0.08 -0.08 -0.08 0.52 
------------------
  r/c   0    1    2    3 
   0 0.32 0.67 0.92    0 
   1 -0.11    0 0.71 0.92 
   2 -0.11 -0.11 0.43 0.62 
------------------
  r/c   0    1    2    3 
   0 0.46 0.75 0.94    0 
   1 0.17    0 0.77 0.94 
   2 -0.14 0.25 0.52 0.72 
------------------
  r/c   0    1    2    3 
   0 0.55 0.77 0.95    0 
   1 0.33    0 0.79 0.95 
   2 0.14 0.38  0.6 0.75 
------------------
  r/c   0    1    2    3 
   0 0.59 0.78 0.95    0 
   1 0.42    0  0.8 0.95 
   2 0.27 0.46 0.63 0.76 
------------------
  r/c   0    1    2    3 
   0 0.61 0.78 0.95    0 
   1 0.46    0  0.8 0.95 
   2 0.35  0.5 0.64 0.77 
------------------
  r/c   0    1    2    3 
   0 0.62 0.78 0.95    0 
   1 0.48    0  0.8 0.95 
   2

In [72]:
def read_policy(grid: Grid, V, gamma=0.9, incumbent_policy=None):
    policy = [[None for _ in range(grid.cols)] for _ in range(grid.rows)]
    for r in range(grid.rows):
        for c in range(grid.cols):
            cell = (r, c)
            if cell in grid.walls or cell in grid.terminals:
                continue
            def q(action):
                result = grid.step(cell, action)
                q_value = 0
                for prob, next_cell, reward in result:
                    q_value += prob * (
                        reward + gamma * V[next_cell[0]][next_cell[1]]
                    )
                return q_value
            new_action = max(grid.actions, key=q)
            if incumbent_policy:
                incumbent_action = max(incumbent_policy[r][c], key=incumbent_policy[r][c].get)
                if q(new_action) - q(incumbent_action) < 1e-9:
                    new_action = incumbent_action
            policy[r][c] = {new_action: 1.0}
    return policy


policy = read_policy(default_grid, V)
policy

[[{'right': 1.0}, {'right': 1.0}, {'right': 1.0}, None],
 [{'up': 1.0}, None, {'up': 1.0}, {'up': 1.0}],
 [{'right': 1.0}, {'right': 1.0}, {'up': 1.0}, {'up': 1.0}]]

In [73]:
render_policy(default_grid, policy)

r/c 0  1  2  3 
 0  →  →  →  1 
 1  ↑  #  ↑  ↑ 
 2  →  →  ↑  ↑ 
------------------


In [74]:
def make_Q(grid: Grid):
    """Q[r][c] is a {action_name: value} dict -- same shape as `policy`.

    All four actions must exist and start EQUAL: control has to be able to
    compare them, and a missing key can never be chosen or learned.
    """
    return [[{a: 0.0 for a in grid.actions} for _ in range(grid.cols)]
            for _ in range(grid.rows)]

In [75]:
def epsilon_greedy(Q, cell, eps, actions, rng):
    """eps-greedy over the Q row at `cell`. Returns an action NAME.

    Random action w.p. eps (EXPLORE), else argmax_a Q[cell][a] (EXPLOIT) with a
    RANDOM tie-break -- at init every action is 0.0, so a fixed tie-break would
    march the agent one direction out of every unexplored cell.
    """
    if rng.random() < eps:
        return actions[int(rng.integers(len(actions)))]
    q = Q[cell[0]][cell[1]]
    best = max(q.values())
    ties = [a for a, v in q.items() if v == best]
    return ties[int(rng.integers(len(ties)))]

In [76]:
ACTIONS = list(default_grid.actions)

Q = make_Q(default_grid)
Q[0][0]['right'] = 1.0          # make 'right' the unique greedy action at (0,0)
rng = np.random.default_rng(1)

n = 40000
print(" eps  | P(greedy 'right') | formula (1-eps)+eps/nA")
print("------+-------------------+-----------------------")
for eps in (0.0, 0.1, 0.3, 1.0):
    picks = [epsilon_greedy(Q, (0, 0), eps, ACTIONS, rng) for _ in range(n)]
    print(f" {eps:.1f}  |      {picks.count('right') / n:.4f}       |"
          f"        {(1 - eps) + eps / len(ACTIONS):.4f}")

# untouched cell: all four tie at 0.0, so the tie-break must spread UNIFORMLY.
# Lock onto one action here and most (cell, action) pairs never get data at all.
picks = [epsilon_greedy(make_Q(default_grid), (1, 0), 0.0, ACTIONS, rng)
         for _ in range(n)]
print("\nuntouched cell, eps=0 ->",
      {a: round(picks.count(a) / n, 3) for a in ACTIONS})

 eps  | P(greedy 'right') | formula (1-eps)+eps/nA
------+-------------------+-----------------------
 0.0  |      1.0000       |        1.0000
 0.1  |      0.9247       |        0.9250
 0.3  |      0.7763       |        0.7750
 1.0  |      0.2485       |        0.2500

untouched cell, eps=0 -> {'up': 0.252, 'right': 0.246, 'down': 0.248, 'left': 0.255}


In [77]:
class ControlSampler(Sampler):
    """Sampler whose reset() returns a FIXED start cell.

    Prediction could use exploring starts to get coverage for free -- that's what
    `Sampler.reset()` does. In control the agent starts where the task starts, and
    eps-greedy is what now has to reach the rest of the grid.
    """

    def __init__(self, env: Grid, rng: np.random.Generator, start=(2, 0)):
        super().__init__(env, rng)
        self.start = start

    def reset(self):
        return self.start

In [78]:
def sarsa(sampler, actions, rng, gamma=0.9, num_episodes=20000, alpha=0.05,
          eps0=1.0, eps_min=0.05, max_steps=1000, episode_returns=None):
    """SARSA -- ON-policy TD control. Uses only sampler.reset()/step(), never grid.step().

    The update consumes exactly (S, A, R, S', A') -- hence the name:

        Q(s,a) <- Q(s,a) + alpha * [ r + gamma*Q(s',a') - Q(s,a) ]

    a' is drawn ONCE by epsilon_greedy and then reused as the next iteration's
    action. That reuse is the whole on-policy story: the action it bootstraps off
    is the action it actually goes on to take, so SARSA evaluates the eps-greedy
    policy it is FOLLOWING, exploration included -- not the greedy one.

    eps anneals eps0 -> eps_min, so behaviour starts as a random walk (coverage) and
    ends near-greedy (which is why max_a Q converges toward V*).

    Pass a list as `episode_returns` to collect the UNDISCOUNTED reward earned per
    training episode -- what the agent actually lived through, as opposed to what its
    final policy would score. The cliff cells below need exactly that distinction.
    """
    grid = sampler.env
    Q = make_Q(grid)
    for ep in range(num_episodes):
        eps = max(eps_min, eps0 * (1 - ep / num_episodes))
        cell = sampler.reset()
        action = epsilon_greedy(Q, cell, eps, actions, rng)
        G = 0.0
        for _ in range(max_steps):
            next_cell, reward, done = sampler.step(cell, action)
            G += reward
            next_action = epsilon_greedy(Q, next_cell, eps, actions, rng)
            # a terminal has no future, so there is nothing to bootstrap off
            bootstrap = 0.0 if done else gamma * Q[next_cell[0]][next_cell[1]][next_action]
            td_error = reward + bootstrap - Q[cell[0]][cell[1]][action]
            Q[cell[0]][cell[1]][action] += alpha * td_error
            cell, action = next_cell, next_action     # reuse a' -- this is the "on-policy"
            if done:
                break
        if episode_returns is not None:
            episode_returns.append(G)
    return Q

In [79]:
def policy_from_Q(grid: Grid, Q):
    """Greedy policy read straight off Q -- argmax_a Q[cell][a], no model needed.
    Compare with `read_policy`, which needs grid.step() to do the same job from V."""
    policy = [[None for _ in range(grid.cols)] for _ in range(grid.rows)]
    for r in range(grid.rows):
        for c in range(grid.cols):
            if (r, c) in grid.walls or (r, c) in grid.terminals:
                continue
            policy[r][c] = {max(Q[r][c], key=Q[r][c].get): 1.0}
    return policy


def V_from_Q(grid: Grid, Q):
    """V(s) = max_a Q(s,a) -- only equals V* once the policy has annealed to greedy."""
    return [[0.0 if (r, c) in grid.walls or (r, c) in grid.terminals
             else max(Q[r][c].values())
             for c in range(grid.cols)] for r in range(grid.rows)]

In [80]:
INNER = [(r, c) for r in range(default_grid.rows) for c in range(default_grid.cols)
         if (r, c) not in default_grid.walls and (r, c) not in default_grid.terminals]


def compare_to_star(name, Q, grid=None, V_star=None, pi_star=None, show=True):
    """Grade a learned Q against value iteration's ground truth.

    Returns (mean signed bias, RMSE). The SIGN matters as much as the magnitude --
    see the on/off-policy cells below -- so we report both, not just RMSE.
    """
    grid = grid if grid is not None else default_grid
    V_star = V_star if V_star is not None else V
    pi_star = pi_star if pi_star is not None else policy

    pol = policy_from_Q(grid, Q)
    V_hat = V_from_Q(grid, Q)
    diffs = [V_hat[r][c] - V_star[r][c] for r, c in INNER]
    bias, rmse = float(np.mean(diffs)), float(np.sqrt(np.mean(np.square(diffs))))

    if show:
        print(f"{name} greedy policy:")
        render_policy(grid, pol)
        print("optimal policy (value iteration):")
        render_policy(grid, pi_star)
        match = sum(max(pol[r][c], key=pol[r][c].get)
                    == max(pi_star[r][c], key=pi_star[r][c].get) for r, c in INNER)
        print(f"actions matching pi*: {match}/{len(INNER)}\n")

        head = f"{name} max_a Q"
        col = max(len(head), 13)                  # widen with the name, keep columns aligned
        print(f" cell  | {head.center(col)} |   V*    |  diff")
        print(f"-------+{'-' * (col + 2)}+---------+--------")
        for r, c in INNER:
            print(f" ({r},{c}) | {f'{V_hat[r][c]:+.4f}'.center(col)} |"
                  f" {V_star[r][c]:+.4f} | {V_hat[r][c] - V_star[r][c]:+.4f}")
        print(f"\nmean bias = {bias:+.4f}   RMSE vs V* = {rmse:.4f}")
    return bias, rmse

In [81]:
sampler = ControlSampler(default_grid, np.random.default_rng(0))
Q_sarsa = sarsa(sampler, ACTIONS, np.random.default_rng(0))

compare_to_star("SARSA", Q_sarsa);

SARSA greedy policy:
r/c 0  1  2  3 
 0  →  →  →  1 
 1  ↑  #  ↑  ↑ 
 2  →  →  ↑  ↑ 
------------------
optimal policy (value iteration):
r/c 0  1  2  3 
 0  →  →  →  1 
 1  ↑  #  ↑  ↑ 
 2  →  →  ↑  ↑ 
------------------
actions matching pi*: 10/10

 cell  | SARSA max_a Q |   V*    |  diff
-------+---------------+---------+--------
 (0,0) |    +0.6037    | +0.6267 | -0.0230
 (0,1) |    +0.7829    | +0.7850 | -0.0021
 (0,2) |    +0.9667    | +0.9496 | +0.0171
 (1,0) |    +0.4754    | +0.5015 | -0.0261
 (1,2) |    +0.8061    | +0.8013 | +0.0048
 (1,3) |    +0.9575    | +0.9496 | +0.0079
 (2,0) |    +0.4144    | +0.4212 | -0.0067
 (2,1) |    +0.5307    | +0.5252 | +0.0056
 (2,2) |    +0.6708    | +0.6537 | +0.0172
 (2,3) |    +0.7881    | +0.7720 | +0.0161

mean bias = +0.0011   RMSE vs V* = 0.0149


In [82]:
def q_learning(sampler, actions, rng, gamma=0.9, num_episodes=20000, alpha=0.05,
               eps0=1.0, eps_min=0.05, max_steps=1000, episode_returns=None):
    """Q-learning -- OFF-policy TD control. Bootstrap off the BEST next action.

        Q(s,a) <- Q(s,a) + alpha * [ r + gamma*max_a' Q(s',a') - Q(s,a) ]
                                            ^^^^^^^^^^^^^^^^^ the only change

    Three differences from `sarsa`, all forced by that max:
      - the action is drawn fresh INSIDE the loop (nothing is carried over),
      - the target uses max(...values()) instead of Q[s'][a'],
      - only `cell` advances, not `(cell, action)`.

    Behaviour is still eps-greedy, but the TARGET evaluates the greedy policy:
    behaviour policy != target policy, which is what "off-policy" means. That max
    is the same Bellman-* backup value_iteration sweeps, applied one sampled
    transition at a time.
    """
    grid = sampler.env
    Q = make_Q(grid)
    for ep in range(num_episodes):
        eps = max(eps_min, eps0 * (1 - ep / num_episodes))
        cell = sampler.reset()
        G = 0.0
        for _ in range(max_steps):
            action = epsilon_greedy(Q, cell, eps, actions, rng)
            next_cell, reward, done = sampler.step(cell, action)
            G += reward
            # a terminal has no future, so there is nothing to bootstrap off
            bootstrap = (
                0.0 if done else gamma * max(Q[next_cell[0]][next_cell[1]].values())
            )
            td_error = reward + bootstrap - Q[cell[0]][cell[1]][action]
            Q[cell[0]][cell[1]][action] += alpha * td_error
            cell = next_cell
            if done:
                break
        if episode_returns is not None:
            episode_returns.append(G)
    return Q

In [83]:
sampler = ControlSampler(default_grid, np.random.default_rng(0))
Q_qlearning = q_learning(sampler, ACTIONS, np.random.default_rng(0))

compare_to_star("Q-learning", Q_qlearning);

Q-learning greedy policy:
r/c 0  1  2  3 
 0  →  →  →  1 
 1  ↑  #  ↑  ↑ 
 2  →  →  ↑  ↑ 
------------------
optimal policy (value iteration):
r/c 0  1  2  3 
 0  →  →  →  1 
 1  ↑  #  ↑  ↑ 
 2  →  →  ↑  ↑ 
------------------
actions matching pi*: 10/10

 cell  | Q-learning max_a Q |   V*    |  diff
-------+--------------------+---------+--------
 (0,0) |      +0.6426       | +0.6267 | +0.0159
 (0,1) |      +0.7802       | +0.7850 | -0.0048
 (0,2) |      +0.9742       | +0.9496 | +0.0246
 (1,0) |      +0.5092       | +0.5015 | +0.0077
 (1,2) |      +0.8043       | +0.8013 | +0.0030
 (1,3) |      +0.9416       | +0.9496 | -0.0080
 (2,0) |      +0.4122       | +0.4212 | -0.0090
 (2,1) |      +0.5367       | +0.5252 | +0.0115
 (2,2) |      +0.6586       | +0.6537 | +0.0049
 (2,3) |      +0.7573       | +0.7720 | -0.0147

mean bias = +0.0031   RMSE vs V* = 0.0121


### The on/off-policy gap, as a measurable number

A single seed can't tell these apart — at seed 0 both look like ±0.02 scatter around `V*`.
Average the **signed** bias over seeds and the two separate cleanly:

- **SARSA is systematically LOW.** With `eps_min=0.05` its target bootstraps off `a'`, the
  action it actually takes — which is a random one 5% of the time. So it converges to the
  value of the *eps-greedy* policy, and that policy genuinely is worse than `pi*`: it walks
  into walls and off the path one step in twenty. SARSA reports that honestly.
- **Q-learning sits ON `V*`.** Its `max` target evaluates the greedy policy no matter how the
  behaviour explored, so exploration cost never enters the estimate.

That gap *is* the on/off-policy distinction, showing up on this gridworld before we ever get
to Cliff Walking. Here it only moves the **values**; on the cliff it changes the **policy**.

The prediction that follows: set `eps_min=0.0` and SARSA's bias should collapse toward
Q-learning's, because once behaviour is greedy `a' == argmax a'` and the two targets are the
same expression.

In [84]:
# ~75s: 10 seeds x 2 algorithms x 2 eps_min settings, 20000 episodes each.
SEEDS = 10


def mean_bias(fn, seeds=SEEDS, **kw):
    """Signed bias of max_a Q against V*, averaged over seeds. Returns (mean, SE)."""
    b = np.array([
        compare_to_star(
            "", fn(ControlSampler(default_grid, np.random.default_rng(s)),
                   ACTIONS, np.random.default_rng(s), **kw), show=False)[0]
        for s in range(seeds)])
    return b.mean(), b.std(ddof=1) / np.sqrt(len(b))


print(f"mean signed bias of max_a Q vs V*, {SEEDS} seeds (+/- 1 SE)")
print(" eps_min |       SARSA        |     Q-learning     |   gap")
print("---------+--------------------+--------------------+---------")
for eps_min in (0.05, 0.0):
    ms, ses = mean_bias(sarsa, eps_min=eps_min)
    mq, seq = mean_bias(q_learning, eps_min=eps_min)
    print(f"   {eps_min:.2f}  |  {ms:+.4f} +/- {ses:.4f} |"
          f"  {mq:+.4f} +/- {seq:.4f} |  {mq - ms:+.4f}")

print("\n eps_min=0.05: SARSA sits several SE BELOW zero -- it is pricing in the 5%")
print("               chance of a random step. Q-learning sits ON zero.")
print(" eps_min=0.00: SARSA's bias collapses toward Q-learning's. With greedy")
print("               behaviour a' IS the argmax, so the two targets coincide.")

mean signed bias of max_a Q vs V*, 10 seeds (+/- 1 SE)
 eps_min |       SARSA        |     Q-learning     |   gap
---------+--------------------+--------------------+---------
   0.05  |  -0.0130 +/- 0.0025 |  +0.0011 +/- 0.0029 |  +0.0142
   0.00  |  -0.0027 +/- 0.0025 |  +0.0032 +/- 0.0027 |  +0.0060

 eps_min=0.05: SARSA sits several SE BELOW zero -- it is pricing in the 5%
               chance of a random step. Q-learning sits ON zero.
 eps_min=0.00: SARSA's bias collapses toward Q-learning's. With greedy
               behaviour a' IS the argmax, so the two targets coincide.


## Cliff Walking — where on/off-policy changes the POLICY, not just the values

Above, the on/off-policy difference only moved the numbers (`-0.0130` vs `+0.0011`). Here it
changes what the agent *does*.

```-
 ·  ·  ·  ·  ·  ·  ·  ·  ·  ·  ·  ·
 ·  ·  ·  ·  ·  ·  ·  ·  ·  ·  ·  ·
 ·  ·  ·  ·  ·  ·  ·  ·  ·  ·  ·  ·
 S  C  C  C  C  C  C  C  C  C  C  G
```

Deterministic moves, every step costs `-1`, `gamma=1.0`. The bottom row between S and G is a
cliff: step on it and you pay `-100` and get teleported back to S, episode still running.

The optimal path skirts the cliff edge along row 2 — 13 steps, the shortest possible. But the
agent behaves eps-greedily, and one random `down` from the edge is a `-100`.

- **SARSA** bootstraps off `a'`, the action it actually takes next — which sometimes *is* that
  fatal step. So the edge genuinely looks bad to it, and it learns a safe detour along the top.
- **Q-learning** bootstraps off `max`, assuming a perfect greedy next move. The edge looks
  fine, so it learns the optimal edge path — and keeps falling off it while exploring.

Note `eps` is held **constant at 0.1** here, not annealed. The whole effect is about what an
agent does when it knows exploration never stops.

In [85]:
class Cliff(Grid):
    """Sutton & Barto Example 6.6. 4x12, deterministic moves, undiscounted (gamma=1).

    Start bottom-left, goal bottom-right; the cells between them are a CLIFF. Stepping
    on one costs -100 and teleports you back to the start WITHOUT ending the episode --
    so falling off is a pure loss, not an escape. Every other step costs -1.

    Subclassing Grid means step() returns the same [(prob, next_cell, reward)] shape,
    so ControlSampler, make_Q, epsilon_greedy, sarsa and q_learning all run on it
    UNCHANGED. If they do, they really were env-agnostic.
    """

    def __init__(self):
        super().__init__(
            rows=4,
            cols=12,
            step_reward=-1.0,
            terminals={(3, 11): -1.0},  # reaching the goal still costs a step
            walls=set(),
            noise=0.0,
        )
        self.start = (3, 0)
        self.cliff = {(3, c) for c in range(1, 11)}

    def cell_repr(self, r, c):
        if (r, c) in self.cliff:
            return 'C'
        if (r, c) == self.start:
            return 'S'
        if (r, c) in self.terminals:
            return 'G'
        return '·'

    def step(self, cell, action):
        dr, dc = self.actions[action]
        nxt = (cell[0] + dr, cell[1] + dc)
        if not (0 <= nxt[0] < self.rows and 0 <= nxt[1] < self.cols):
            nxt = cell                                   # walk into the border, stay put
        if nxt in self.cliff:
            return [(1.0, self.start, -100.0)]           # deterministic -> a single outcome
        return [(1.0, nxt, self.terminals.get(nxt, self.step_reward))]


def render_cliff(cliff, Q):
    """Arrow map, with S / C / G drawn instead of an arrow."""
    pol = policy_from_Q(cliff, Q)
    for r in range(cliff.rows):
        row = ""
        for c in range(cliff.cols):
            if (r, c) in cliff.cliff or (r, c) == cliff.start or (r, c) in cliff.terminals:
                row += " " + cliff.cell_repr(r, c)
            else:
                row += " " + cliff.arrows[max(pol[r][c], key=pol[r][c].get)]
        print(row)
    print()


def greedy_path(cliff, Q, max_len=100):
    """Walk the greedy (eps=0) policy from the start.
    Returns (steps, min row reached, reached_goal). Lower row = closer to the top = safer."""
    cell, rows = cliff.start, []
    for i in range(max_len):
        rows.append(cell[0])
        if cell in cliff.terminals:
            return i, min(rows), True
        action = max(Q[cell[0]][cell[1]], key=Q[cell[0]][cell[1]].get)
        nxt = cliff.step(cell, action)[0][1]
        if nxt == cell:                                  # stuck against a border
            return i, min(rows), False
        cell = nxt
    return max_len, min(rows), False


cliff = Cliff()
CLIFF_ACTIONS = list(cliff.actions)
cliff.render()

# sanity-check the dynamics before trusting anything learned on them
probe = ControlSampler(cliff, np.random.default_rng(0), start=cliff.start)
print("\nright from start (3,0) -> onto the cliff:", probe.step((3, 0), 'right'))
print("up    from start (3,0)                  :", probe.step((3, 0), 'up'))
print("down  from (2,5)       -> onto the cliff:", probe.step((2, 5), 'down'))
print("down  from (2,11)      -> into the goal :", probe.step((2, 11), 'down'))
print("up    from (0,0)       -> border, stays :", probe.step((0, 0), 'up'))

 r/c  0   1   2   3   4   5   6   7   8   9  10  11 
  0   ·   ·   ·   ·   ·   ·   ·   ·   ·   ·   ·   · 
  1   ·   ·   ·   ·   ·   ·   ·   ·   ·   ·   ·   · 
  2   ·   ·   ·   ·   ·   ·   ·   ·   ·   ·   ·   · 
  3   S   C   C   C   C   C   C   C   C   C   C   G 

right from start (3,0) -> onto the cliff: ((3, 0), -100.0, False)
up    from start (3,0)                  : ((2, 0), -1.0, False)
down  from (2,5)       -> onto the cliff: ((3, 0), -100.0, False)
down  from (2,11)      -> into the goal : ((3, 11), -1.0, True)
up    from (0,0)       -> border, stays : ((0, 0), -1.0, False)


In [86]:
# Constant eps=0.1, NO annealing -- the divergence only exists while exploration is live.
CLIFF_KW = dict(gamma=1.0, num_episodes=500, alpha=0.5,
                eps0=0.1, eps_min=0.1, max_steps=1000)

returns_sarsa, returns_q = [], []
Q_cliff_sarsa = sarsa(
    ControlSampler(cliff, np.random.default_rng(0), start=cliff.start),
    CLIFF_ACTIONS, np.random.default_rng(0),
    episode_returns=returns_sarsa, **CLIFF_KW)
Q_cliff_q = q_learning(
    ControlSampler(cliff, np.random.default_rng(0), start=cliff.start),
    CLIFF_ACTIONS, np.random.default_rng(0),
    episode_returns=returns_q, **CLIFF_KW)

print("SARSA (on-policy) -- hugs the TOP, far from the cliff:")
render_cliff(cliff, Q_cliff_sarsa)
print("Q-learning (off-policy) -- hugs the EDGE, the optimal path:")
render_cliff(cliff, Q_cliff_q)

steps_s, row_s, ok_s = greedy_path(cliff, Q_cliff_sarsa)
steps_q, row_q, ok_q = greedy_path(cliff, Q_cliff_q)
print(f"greedy path   SARSA      : {steps_s:3d} steps, closest row to cliff = {row_s}, "
      f"reached goal = {ok_s}")
print(f"              Q-learning : {steps_q:3d} steps, closest row to cliff = {row_q}, "
      f"reached goal = {ok_q}   <- shorter = optimal")

print(f"\nONLINE return during training (mean of last 100 episodes):")
print(f"  SARSA      {np.mean(returns_sarsa[-100:]):+8.1f}   (rarely falls off)")
print(f"  Q-learning {np.mean(returns_q[-100:]):+8.1f}   (keeps falling off while exploring)")
print("\n The twist: Q-learning's FINAL policy is optimal, yet it EARNS LESS while")
print(" learning. Its target assumes a perfect greedy next move, so it never prices")
print(" in the eps-chance of stepping off the edge it is walking along. SARSA's target")
print(" bootstraps off the action it really takes, feels that risk, and buys safety")
print(" for 4 extra steps. Optimal policy != best behaviour while you must explore.")

SARSA (on-policy) -- hugs the TOP, far from the cliff:
 → → → → → → → → → → ↓ ↓
 ↑ ↑ ↑ → ↑ ↑ → → ↑ ↑ → ↓
 ↑ ↑ ↑ → ↑ ← ← ↑ ↑ → → ↓
 S C C C C C C C C C C G

Q-learning (off-policy) -- hugs the EDGE, the optimal path:
 ↓ ↑ ↑ → → → ↓ → ↓ → → ↓
 → → → ↓ ↓ ↓ → ↓ → → ↓ ↓
 → → → → → → → → → → → ↓
 S C C C C C C C C C C G

greedy path   SARSA      :  17 steps, closest row to cliff = 0, reached goal = True
              Q-learning :  13 steps, closest row to cliff = 2, reached goal = True   <- shorter = optimal

ONLINE return during training (mean of last 100 episodes):
  SARSA         -26.2   (rarely falls off)
  Q-learning    -30.1   (keeps falling off while exploring)

 The twist: Q-learning's FINAL policy is optimal, yet it EARNS LESS while
 learning. Its target assumes a perfect greedy next move, so it never prices
 in the eps-chance of stepping off the edge it is walking along. SARSA's target
 bootstraps off the action it really takes, feels that risk, and buys safety
 for 4 extra steps. 

### The control: is the safe path really about eps?

The claim above is that SARSA detours *because* exploration is permanent. The way to test it
is to shrink eps and watch the detour disappear. It does — but only with **constant** eps.

**Annealing eps to 0 does NOT collapse the two policies.** Measured: 2000, 10000 and 50000
episodes, alpha 0.5 and 0.1, all leave SARSA on the top row at 17 steps. Training a further
10000 episodes at eps=0 on top of the annealed `Q` changes nothing either.

The reason is worth internalising, because it is a real failure mode and not a quirk of this
env. Look at the edge cell `(2,5)` after an annealed run:

```-
{'up': -652.93, 'right': -799.86, 'down': -1160.71, 'left': -984.08}
```

Those values were learned while eps was large and the agent was falling off constantly. Once
eps hits 0, the agent takes only the argmax in every state — so `right` at `(2,5)` is **never
sampled again**, and its `-799` is frozen forever. The policy can't improve toward the edge
because the evidence that would fix it can never be collected.

This is the GLIE condition biting: convergence to `Q*` needs every `(s,a)` visited infinitely
often, and eps->0 on a fixed-start task kills exactly that. Exploring starts don't rescue it
either (also measured) — random starts fix *state* coverage, but within an episode eps=0
still means only one action per state ever gets tried.

So the honest control is a sweep over **constant** eps, below.

In [87]:
# ~10s. Sweep CONSTANT eps down to 0 and watch SARSA's path walk toward the edge.
print("constant eps (no annealing), 8000 episodes, alpha=0.1")
print("  eps  |     SARSA      |   Q-learning   | SARSA row")
print("-------+----------------+----------------+-----------")
for eps in (0.10, 0.05, 0.02, 0.00):
    kw = dict(gamma=1.0, num_episodes=8000, alpha=0.1,
              eps0=eps, eps_min=eps, max_steps=1000)
    Qs_ = sarsa(ControlSampler(cliff, np.random.default_rng(0), start=cliff.start),
                CLIFF_ACTIONS, np.random.default_rng(0), **kw)
    Qq_ = q_learning(ControlSampler(cliff, np.random.default_rng(0), start=cliff.start),
                     CLIFF_ACTIONS, np.random.default_rng(0), **kw)
    (ss, sr, _), (qs, qr, _) = greedy_path(cliff, Qs_), greedy_path(cliff, Qq_)
    print(f"  {eps:.2f} |  {ss:2d} steps row {sr} |  {qs:2d} steps row {qr} |"
          f"  {'top' if sr == 0 else 'middle' if sr == 1 else 'EDGE'}")

print("\n SARSA's path slides down toward the cliff as eps shrinks -- top row, then")
print(" middle, then at eps=0 it lands exactly on Q-learning's 13-step edge path.")
print(" Q-learning is pinned at the edge the whole way: its max target never saw")
print(" the exploration risk in the first place. The safe path was never SARSA")
print(" being worse at RL -- it was SARSA correctly pricing the eps it was given.")

constant eps (no annealing), 8000 episodes, alpha=0.1
  eps  |     SARSA      |   Q-learning   | SARSA row
-------+----------------+----------------+-----------
  0.10 |  17 steps row 0 |  13 steps row 2 |  top
  0.05 |  15 steps row 1 |  13 steps row 2 |  middle
  0.02 |  15 steps row 1 |  13 steps row 2 |  middle
  0.00 |  13 steps row 2 |  13 steps row 2 |  EDGE

 SARSA's path slides down toward the cliff as eps shrinks -- top row, then
 middle, then at eps=0 it lands exactly on Q-learning's 13-step edge path.
 Q-learning is pinned at the edge the whole way: its max target never saw
 the exploration risk in the first place. The safe path was never SARSA
 being worse at RL -- it was SARSA correctly pricing the eps it was given.


## Expected SARSA — the third algorithm, and the one that unifies the other two

All three algorithms are the same update. They differ only in **which next-action value they
bootstrap off**:

```-
SARSA            r + gamma * Q[s'][a']                 a' ~ eps-greedy   (ONE SAMPLE)
Expected SARSA   r + gamma * SUM_a' pi(a'|s') Q[s'][a']    the EXPECTATION over that pi
Q-learning       r + gamma * max_a' Q[s'][a']              pi = greedy
```

Read the middle line and then the last one. If `pi` is greedy, all the probability mass sits
on the argmax and the sum *collapses to the max* — so **Q-learning is Expected SARSA with a
greedy target policy**. The cell below proves that by running both and diffing the tables.

Two things fall out of this:

1. **On/off-policy is not a binary.** It's a choice of which `pi` you average under. SARSA
   (behaviour policy) and Q-learning (greedy) are the two endpoints of one dial, `target_eps`.
2. **SARSA's target carries variance that Expected SARSA's doesn't.** SARSA draws one `a'` and
   uses its `Q` whole; Expected SARSA averages over all of them exactly. Same expected value,
   less noise — which is why it tolerates much larger `alpha`.

The cost is `nA` lookups per step instead of 1 — irrelevant here, and the reason Expected
SARSA is usually the right default in tabular settings.

In [88]:
def expected_q(Q, cell, eps, actions):
    """E_{a ~ eps-greedy(eps)}[ Q[cell][a] ] -- computed EXACTLY, nothing sampled.

    Must mirror epsilon_greedy's distribution precisely, random tie-break included:
    every action gets eps/nA, and the remaining (1-eps) greedy mass is split evenly
    among the tied argmax actions.
    """
    q = Q[cell[0]][cell[1]]
    best = max(q.values())
    ties = [a for a in actions if q[a] == best]
    p_explore = eps / len(actions)
    p_greedy = (1.0 - eps) / len(ties)
    return sum(q[a] * (p_explore + (p_greedy if a in ties else 0.0)) for a in actions)


def expected_sarsa(sampler, actions, rng, gamma=0.9, num_episodes=20000, alpha=0.05,
                   eps0=1.0, eps_min=0.05, max_steps=1000, target_eps=None,
                   episode_returns=None):
    """Expected SARSA -- bootstrap off the EXPECTED next-action value.

        Q(s,a) <- Q(s,a) + alpha * [ r + gamma * SUM_a' pi(a'|s') Q(s',a') - Q(s,a) ]

    Identical to sarsa() except the target averages over a' instead of sampling one.
    Same expectation, none of the sampling variance -- so it tolerates much larger alpha.

    `target_eps` picks the TARGET policy, independently of the eps used to behave:
      None -> the behaviour eps  => on-policy  (Expected SARSA proper)
      0.0  -> greedy target      => Q-LEARNING, exactly. Off-policy is just a
              different pi in the same sum, not a different algorithm.
    """
    grid = sampler.env
    Q = make_Q(grid)
    for ep in range(num_episodes):
        eps = max(eps_min, eps0 * (1 - ep / num_episodes))
        cell = sampler.reset()
        G = 0.0
        for _ in range(max_steps):
            action = epsilon_greedy(Q, cell, eps, actions, rng)
            next_cell, reward, done = sampler.step(cell, action)
            G += reward
            te = eps if target_eps is None else target_eps
            bootstrap = (0.0 if done
                         else gamma * expected_q(Q, next_cell, te, actions))
            td_error = reward + bootstrap - Q[cell[0]][cell[1]][action]
            Q[cell[0]][cell[1]][action] += alpha * td_error
            cell = next_cell
            if done:
                break
        if episode_returns is not None:
            episode_returns.append(G)
    return Q


# CHECK A: does expected_q really equal the mean of what epsilon_greedy samples?
rng_c = np.random.default_rng(3)
Qprobe = make_Q(default_grid)
for a, v in zip(ACTIONS, [0.3, 0.9, -0.2, 0.5]):
    Qprobe[2][0][a] = v
print(" eps | expected_q (exact) | sampled mean of Q[s'][a']")
print("-----+--------------------+--------------------------")
for eps in (0.0, 0.1, 0.5, 1.0):
    samp = np.mean([Qprobe[2][0][epsilon_greedy(Qprobe, (2, 0), eps, ACTIONS, rng_c)]
                    for _ in range(200000)])
    print(f" {eps:.1f} |      {expected_q(Qprobe, (2, 0), eps, ACTIONS):+.4f}       |"
          f"          {samp:+.4f}")

# CHECK B: target_eps=0.0 must reproduce q_learning BIT FOR BIT. Both consume the
# rng identically (one epsilon_greedy call per step), so the tables must match exactly.
kwc = dict(num_episodes=3000, alpha=0.05)
Qe = expected_sarsa(ControlSampler(default_grid, np.random.default_rng(0)), ACTIONS,
                    np.random.default_rng(0), target_eps=0.0, **kwc)
Qq = q_learning(ControlSampler(default_grid, np.random.default_rng(0)), ACTIONS,
                np.random.default_rng(0), **kwc)
worst = max(abs(Qe[r][c][a] - Qq[r][c][a])
            for r, c in INNER for a in ACTIONS)
print(f"\nCHECK B  Expected SARSA(target_eps=0) vs q_learning: max |diff| = {worst:.3e}")
print("         Q-learning IS Expected SARSA with a greedy target policy.")

 eps | expected_q (exact) | sampled mean of Q[s'][a']
-----+--------------------+--------------------------
 0.0 |      +0.9000       |          +0.9000
 0.1 |      +0.8475       |          +0.8482
 0.5 |      +0.6375       |          +0.6378
 1.0 |      +0.3750       |          +0.3748

CHECK B  Expected SARSA(target_eps=0) vs q_learning: max |diff| = 0.000e+00
         Q-learning IS Expected SARSA with a greedy target policy.


In [89]:
# ~90s. Sutton & Barto Figure 6.3: online return vs alpha on the cliff, 10 seeds.
# SARSA's target carries the variance of SAMPLING a'; Expected SARSA's doesn't.
# So SARSA should have a sweet spot in alpha and get worse past it, while
# Expected SARSA should keep improving as alpha rises.

def cliff_online(fn, alpha, seeds=10, episodes=200):
    """Mean undiscounted return per training episode, averaged over seeds."""
    out = []
    for s in range(seeds):
        R = []
        fn(ControlSampler(cliff, np.random.default_rng(s), start=cliff.start),
           CLIFF_ACTIONS, np.random.default_rng(s), gamma=1.0, num_episodes=episodes,
           alpha=alpha, eps0=0.1, eps_min=0.1, max_steps=1000, episode_returns=R)
        out.append(np.mean(R))
    return float(np.mean(out))


print("CLIFF: mean ONLINE return over the first 200 episodes (10 seeds, eps=0.1)")
print(" alpha |  SARSA  | Expected SARSA | Q-learning")
print("-------+---------+----------------+-----------")
for alpha in (0.1, 0.3, 0.5, 0.8, 1.0):
    a = cliff_online(sarsa, alpha)
    b = cliff_online(expected_sarsa, alpha)
    c = cliff_online(q_learning, alpha)
    print(f"  {alpha:.1f}  | {a:7.1f} |    {b:7.1f}     |  {c:7.1f}")

print("\n Expected SARSA beats SARSA at EVERY alpha, and the gap widens as alpha grows.")
print(" SARSA peaks around alpha=0.5 and then degrades -- at alpha=1.0 each update")
print(" throws away the old estimate and adopts one noisy sample of Q[s'][a'] whole.")
print(" Expected SARSA has no such sample to be noisy: it averages over a' exactly,")
print(" so alpha=1.0 is safe and it just learns faster. Q-learning stays worst on this")
print(" metric throughout -- it is optimising the wrong thing for ONLINE performance.")

CLIFF: mean ONLINE return over the first 200 episodes (10 seeds, eps=0.1)
 alpha |  SARSA  | Expected SARSA | Q-learning
-------+---------+----------------+-----------
  0.1  |   -91.0 |      -86.4     |    -99.7
  0.3  |   -55.1 |      -47.3     |    -69.6
  0.5  |   -49.9 |      -39.2     |    -67.1
  0.8  |   -56.6 |      -34.4     |    -64.4
  1.0  |   -61.7 |      -32.2     |    -63.2

 Expected SARSA beats SARSA at EVERY alpha, and the gap widens as alpha grows.
 SARSA peaks around alpha=0.5 and then degrades -- at alpha=1.0 each update
 throws away the old estimate and adopts one noisy sample of Q[s'][a'] whole.
 Expected SARSA has no such sample to be noisy: it averages over a' exactly,
 so alpha=1.0 is safe and it just learns faster. Q-learning stays worst on this
 metric throughout -- it is optimising the wrong thing for ONLINE performance.


### Does Expected SARSA estimate `V*` any better than SARSA?

The alpha sweep only compared ONLINE return. This asks the other question: how accurate is the
`Q` it ends up with?

**Prediction.** On-policy Expected SARSA and SARSA share a *fixed point* — both evaluate the
eps-greedy policy — so their **bias** should match. But Expected SARSA's target carries no
sampling noise, so its **RMSE** should be lower.

Half right. The bias prediction holds. The RMSE prediction is **wrong** — measured below,
Expected SARSA's RMSE comes out slightly *worse*. At `alpha=0.05` the running average has
already suppressed target noise, so what remains is the eps-greedy fixed-point bias, and
averaging over `a'` cannot touch that. Expected SARSA's variance advantage is
**alpha-dependent**: it appears only where alpha is large, which is exactly where the cliff
sweep found it.

`target_eps=0.0` is also here as a control — it should reproduce Q-learning digit for digit.

In [90]:
# ~50s. All four graded against V* on the plain gridworld, 8 seeds.
# Both eps_min rows, so the annealing question is covered too.
def grid_stats(fn, seeds=8, **kw):
    """(mean bias, SE, mean RMSE) of max_a Q vs V*, over seeds."""
    out = np.array([
        compare_to_star("", fn(ControlSampler(default_grid, np.random.default_rng(s)),
                               ACTIONS, np.random.default_rng(s),
                               num_episodes=8000, **kw), show=False)
        for s in range(seeds)])
    return out[:, 0].mean(), out[:, 0].std(ddof=1) / np.sqrt(seeds), out[:, 1].mean()


VARIANTS = (("SARSA", sarsa, {}),
            ("Expected SARSA (on)", expected_sarsa, {}),
            ("Expected SARSA te=0", expected_sarsa, dict(target_eps=0.0)),
            ("Q-learning", q_learning, {}))

print("mean signed bias and RMSE vs V*, 8 seeds, 8000 episodes, alpha=0.05")
print(" eps_min | algorithm            |    mean bias       |  RMSE")
print("---------+----------------------+--------------------+--------")
for eps_min in (0.05, 0.0):
    for name, fn, extra in VARIANTS:
        b, se, r = grid_stats(fn, eps_min=eps_min, **extra)
        print(f"  {eps_min:.2f}   | {name:20} | {b:+.4f} +/- {se:.4f}  | {r:.4f}")
    print()

print(" eps_min=0.05: Expected SARSA (on-policy) sits on SARSA's bias, NOT between the")
print("   two -- they share a fixed point, so averaging over a' changes the noise, never")
print("   the thing being estimated. Its RMSE is NOT better: at alpha=0.05 the target")
print("   noise is already averaged away and only fixed-point bias is left.")
print("   te=0 matches Q-learning digit for digit -- the identity check at scale.")
print(" eps_min=0.00: all four converge. Greedy behaviour makes every target the same")
print("   expression, so the algorithms only differ while exploration is live.")

mean signed bias and RMSE vs V*, 8 seeds, 8000 episodes, alpha=0.05
 eps_min | algorithm            |    mean bias       |  RMSE
---------+----------------------+--------------------+--------
  0.05   | SARSA                | -0.0127 +/- 0.0033  | 0.0213
  0.05   | Expected SARSA (on)  | -0.0180 +/- 0.0037  | 0.0259
  0.05   | Expected SARSA te=0  | -0.0019 +/- 0.0031  | 0.0139
  0.05   | Q-learning           | -0.0019 +/- 0.0031  | 0.0139

  0.00   | SARSA                | -0.0077 +/- 0.0021  | 0.0193
  0.00   | Expected SARSA (on)  | -0.0063 +/- 0.0023  | 0.0168
  0.00   | Expected SARSA te=0  | -0.0046 +/- 0.0018  | 0.0158
  0.00   | Q-learning           | -0.0046 +/- 0.0018  | 0.0158

 eps_min=0.05: Expected SARSA (on-policy) sits on SARSA's bias, NOT between the
   two -- they share a fixed point, so averaging over a' changes the noise, never
   the thing being estimated. Its RMSE is NOT better: at alpha=0.05 the target
   noise is already averaged away and only fixed-point bias i

### Turning the `target_eps` dial

`target_eps` is the one number separating the three algorithms, and we have not yet turned it.
Below, **behaviour is pinned at eps=0.1 for every run** — same data distribution, same alpha,
same episode budget, same seeds. Only the TARGET policy changes.

**Prediction.** `target_eps=0.1` should reproduce SARSA's safe path, `target_eps=0.0` should
reproduce Q-learning's edge path, and the middle should slide between them.

Two things it gets wrong, both measured in the next two cells:

1. **It is a threshold, not a slide.** Every seed sits on row 1 above `te≈0.008` and every seed
   sits on the edge below it. There is no intermediate regime — and the flip point is
   predictable from the reward function alone (arithmetic in the second cell).
2. **`target_eps=0.1` does NOT reproduce SARSA.** It lands on **row 1**, while SARSA at the
   same eps lands on **row 0** — one row further from the cliff than pricing the risk exactly
   would justify. So part of SARSA's famous caution is the *variance* of its sampled target,
   not the eps it is pricing in.

In [91]:
# ~25s. All four on the cliff under ONE protocol: eps=0.1 constant, alpha=0.5,
# 500 episodes, 12 seeds. Same sampler seeds, same rng seeds, same everything.
from collections import Counter


def cliff_policy_stats(fn, seeds=12, **kw):
    """Row histogram + goal-reaching rate + mean path length of the GREEDY policy."""
    rows, steps, reached = [], [], 0
    for s in range(seeds):
        Q_ = fn(ControlSampler(cliff, np.random.default_rng(s), start=cliff.start),
                CLIFF_ACTIONS, np.random.default_rng(s), gamma=1.0, num_episodes=500,
                alpha=0.5, eps0=0.1, eps_min=0.1, max_steps=1000, **kw)
        st, rw, ok = greedy_path(cliff, Q_)
        rows.append(rw)
        reached += ok
        if ok:
            steps.append(st)
    return dict(sorted(Counter(rows).items())), reached, seeds, float(np.mean(steps))


print("cliff, eps=0.1 constant, alpha=0.5, 500 episodes, 12 seeds -- ONE protocol")
print(" algorithm             | closest row {row: seeds} | reached goal | steps")
print("-----------------------+--------------------------+--------------+-------")
for name, fn, extra in (("SARSA", sarsa, {}),
                        ("Expected SARSA te=0.1", expected_sarsa, dict(target_eps=0.1)),
                        ("Expected SARSA te=0.0", expected_sarsa, dict(target_eps=0.0)),
                        ("Q-learning", q_learning, {})):
    hist, ok, n, mean_steps = cliff_policy_stats(fn, **extra)
    print(f" {name:21} | {str(hist):24} |    {ok:2d}/{n}     | {mean_steps:5.1f}")

print("\n Unanimous, 12/12 seeds each -- these are not near-ties.")
print("   SARSA           -> row 0, the TOP row, 17.4 steps")
print("   Exp SARSA te=0.1-> row 1, 15.0 steps  (same eps SARSA behaves under!)")
print("   Exp SARSA te=0.0-> row 2, 13.0 steps  == Q-learning, exactly")
print("\n Rows 0 and 1 are BOTH one exploratory step from safety -- from row 1 a random")
print(" 'down' lands on row 2, which is still safe. Row 1 is simply shorter, so pricing")
print(" the eps=0.1 risk EXACTLY says row 1. SARSA goes one row further, and pays for it:")
print(" its greedy policy doesn't even reach the goal in 2 of 12 seeds. Sampling one a'")
print(" at alpha=0.5 occasionally draws the -100 and over-corrects. Part of SARSA's")
print(" famous caution is target VARIANCE, not the risk it is reasoning about.\n")

print("SARSA (seed 0):")
render_cliff(cliff, sarsa(
    ControlSampler(cliff, np.random.default_rng(0), start=cliff.start), CLIFF_ACTIONS,
    np.random.default_rng(0), gamma=1.0, num_episodes=500, alpha=0.5,
    eps0=0.1, eps_min=0.1, max_steps=1000))
print("Expected SARSA, target_eps=0.1 -- same eps, one row lower:")
render_cliff(cliff, expected_sarsa(
    ControlSampler(cliff, np.random.default_rng(0), start=cliff.start), CLIFF_ACTIONS,
    np.random.default_rng(0), gamma=1.0, num_episodes=500, alpha=0.5,
    eps0=0.1, eps_min=0.1, max_steps=1000, target_eps=0.1))
print("Expected SARSA, target_eps=0.0 -- the edge, i.e. Q-learning:")
render_cliff(cliff, expected_sarsa(
    ControlSampler(cliff, np.random.default_rng(0), start=cliff.start), CLIFF_ACTIONS,
    np.random.default_rng(0), gamma=1.0, num_episodes=500, alpha=0.5,
    eps0=0.1, eps_min=0.1, max_steps=1000, target_eps=0.0))

cliff, eps=0.1 constant, alpha=0.5, 500 episodes, 12 seeds -- ONE protocol
 algorithm             | closest row {row: seeds} | reached goal | steps
-----------------------+--------------------------+--------------+-------
 SARSA                 | {0: 12}                  |    10/12     |  17.4
 Expected SARSA te=0.1 | {1: 12}                  |    12/12     |  15.0
 Expected SARSA te=0.0 | {2: 12}                  |    12/12     |  13.0
 Q-learning            | {2: 12}                  |    12/12     |  13.0

 Unanimous, 12/12 seeds each -- these are not near-ties.
   SARSA           -> row 0, the TOP row, 17.4 steps
   Exp SARSA te=0.1-> row 1, 15.0 steps  (same eps SARSA behaves under!)
   Exp SARSA te=0.0-> row 2, 13.0 steps  == Q-learning, exactly

 Rows 0 and 1 are BOTH one exploratory step from safety -- from row 1 a random
 'down' lands on row 2, which is still safe. Row 1 is simply shorter, so pricing
 the eps=0.1 risk EXACTLY says row 1. SARSA goes one row further, and pays fo

In [92]:
# ~40s. Now sweep target_eps and find where the policy flips. Behaviour still eps=0.1.
print("cliff, behaviour eps = 0.1 fixed, alpha=0.5, 500 episodes, 8 seeds")
print(" target_eps | mean steps | mean row | seeds on the EDGE | online (last 100)")
print("------------+------------+----------+-------------------+------------------")
for te in (0.10, 0.02, 0.008, 0.005, 0.00):
    steps_l, rows_l, ret_l = [], [], []
    for s in range(8):
        R = []
        Q_ = expected_sarsa(
            ControlSampler(cliff, np.random.default_rng(s), start=cliff.start),
            CLIFF_ACTIONS, np.random.default_rng(s), gamma=1.0, num_episodes=500,
            alpha=0.5, eps0=0.1, eps_min=0.1, max_steps=1000,
            target_eps=te, episode_returns=R)
        st, rw, _ = greedy_path(cliff, Q_)
        steps_l.append(st); rows_l.append(rw); ret_l.append(np.mean(R[-100:]))
    print(f"    {te:.3f}   |    {np.mean(steps_l):4.1f}    |   {np.mean(rows_l):.1f}    |"
          f"       {sum(r == 2 for r in rows_l)}/8         |      {np.mean(ret_l):+7.1f}")

print("\n A THRESHOLD, not a slide -- 0/8 seeds on the edge above ~0.008, 8/8 below.")
print(" And the flip point follows from the reward function alone:")
print("   detour via row 1  = 2 extra steps                  = -2")
print("   edge path risk   ~ 10 edge cells * (te/4) * (-100) = -250*te")
print("   equal when te = 2/250 = 0.008   <- exactly where the table flips.")
print("\n The agent is not being 'cautious' or 'brave'. It is solving an arithmetic")
print(" problem about a risk WE specify via target_eps, and it switches the instant the")
print(" sign flips. SARSA and Q-learning are the two ends of this one number.")

cliff, behaviour eps = 0.1 fixed, alpha=0.5, 500 episodes, 8 seeds
 target_eps | mean steps | mean row | seeds on the EDGE | online (last 100)
------------+------------+----------+-------------------+------------------
    0.100   |    15.0    |   1.0    |       0/8         |        -20.7
    0.020   |    15.0    |   1.0    |       0/8         |        -21.0
    0.008   |    15.0    |   1.0    |       0/8         |        -23.4
    0.005   |    13.0    |   2.0    |       8/8         |        -43.4
    0.000   |    13.0    |   2.0    |       8/8         |        -48.0

 A THRESHOLD, not a slide -- 0/8 seeds on the edge above ~0.008, 8/8 below.
 And the flip point follows from the reward function alone:
   detour via row 1  = 2 extra steps                  = -2
   edge path risk   ~ 10 edge cells * (te/4) * (-100) = -250*te
   equal when te = 2/250 = 0.008   <- exactly where the table flips.

 The agent is not being 'cautious' or 'brave'. It is solving an arithmetic
 problem about a risk